# Stage 1C — Frozen PIT Momentum

Momentum is frozen as two raw components, not a score: `Momentum_raw = [R_12M_AdjustedClose, R_1M_AdjustedClose]`. It represents medium-term price/total-return-like movement plus recent continuation/deterioration. Decision is at XNYS open `t`; the information cutoff is the prior eligible XNYS close.

**Limitation:** current historical adjusted-close data has no evidence of the exact vintage at which each historical adjustment was available. The experiment accepts this series under that explicit limitation; Momentum does not have the same strict PIT provenance as SEC fundamentals.


In [ ]:
from __future__ import annotations
import json
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import exchange_calendars as xcals
ROOT=next(p for p in [Path.cwd(),*Path.cwd().parents] if (p/'data/raw/stage_0/manifest.json').exists()); sys.path.insert(0,str(ROOT/'src'))
from pit_momentum import XNYS, momentum_as_of
manifest=json.loads((ROOT/'data/raw/stage_0/manifest.json').read_text(encoding='utf-8')); prices={}
for item in manifest:
    if item['endpoint']=='TIME_SERIES_DAILY_ADJUSTED' and item['request_status']=='HTTP_200':
        raw=json.loads((ROOT/item['local_raw_file']).read_text(encoding='utf-8'))['Time Series (Daily)']
        f=pd.DataFrame.from_dict(raw,orient='index').rename(columns={'4. close':'close','5. adjusted close':'adjusted_close','7. dividend amount':'dividend_amount','8. split coefficient':'split_coefficient'}); f.index=pd.DatetimeIndex(f.index,name='price_date')
        prices[item['symbol']]=f.apply(pd.to_numeric).sort_index()
def opening(session): return XNYS.session_open(pd.Timestamp(session))


## Calendar/XNYS-aware implementation

The reusable implementation maps `cutoff - 1 calendar year` and `cutoff - 1 calendar month` to the last XNYS session on or before each anchor. It does not use 252 or 21 rows. The returned fields expose cutoff, current adjusted close, both reference dates/prices, and both raw returns.


In [ ]:
# AAPL: strong 12M movement and positive recent continuation.
aapl=momentum_as_of(prices['AAPL'],'AAPL',opening('2007-10-09'))
assert aapl.status=='VALID' and aapl.momentum_12m>0 and aapl.momentum_1m>0
manual_12=aapl.current_adjusted_close/aapl.twelve_month_reference_adjusted_close-1; manual_1=aapl.current_adjusted_close/aapl.one_month_reference_adjusted_close-1
np.testing.assert_allclose([aapl.momentum_12m,aapl.momentum_1m],[manual_12,manual_1])
pd.DataFrame([aapl.__dict__])


In [ ]:
# Existing audit examples: positive 12M but deterioration; weak 12M but rebound; similar 12M / different recent path.
cases=[('XOM','2007-11-06'),('WMT','2007-10-09'),('AAPL','2008-12-23'),('UNH','2008-12-23')]
results=[momentum_as_of(prices[i],i,opening(d)) for i,d in cases]
table=pd.DataFrame([r.__dict__ for r in results])[['issuer','decision_time','momentum_12m','momentum_1m','status']]
assert table.loc[0,'momentum_12m']>0 and table.loc[0,'momentum_1m']<0
assert table.loc[1,'momentum_12m']<0 and table.loc[1,'momentum_1m']>0
assert abs(table.loc[2,'momentum_12m']-table.loc[3,'momentum_12m'])<.02 and abs(table.loc[2,'momentum_1m']-table.loc[3,'momentum_1m'])>.08
table


## Corporate-action validation and exclusions

AAPL's 4-for-1 split on 2020-08-31 makes raw close drop mechanically by about 75%; adjusted close does not. The frozen module uses only adjusted close, so the split cannot create that artificial Momentum return. Proximity-to-high was informative in the audit but is not selected. Classical 12-1 is rejected only for this experiment because it removes recent movement, which this economic definition explicitly retains.


In [ ]:
aapl_prices=prices['AAPL']; raw_split=aapl_prices.loc['2020-08-31','close']/aapl_prices.loc['2020-08-28','close']-1; adjusted_split=aapl_prices.loc['2020-08-31','adjusted_close']/aapl_prices.loc['2020-08-28','adjusted_close']-1
split_result=momentum_as_of(aapl_prices,'AAPL',opening('2020-09-01'))
assert raw_split<-.70 and adjusted_split>-.10 and split_result.momentum_12m>-.10
{'raw_close_split_return':raw_split,'adjusted_close_split_return':adjusted_split,'frozen_R_12M':split_result.momentum_12m}


No scalar Momentum, proximity component, 12-1 component, ranking, normalization, `X_char`, `w_raw`, `F_risk`, portfolio, performance test, or backtest is constructed here.
